# PSX data quality and exploration
This notebook reads the persistent master dataset through project configuration. It does not fetch live data or duplicate pipeline logic.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)

Project root: /Users/m.abdulbasit/Downloads/virtual-trader
Python executable: /Users/m.abdulbasit/Downloads/virtual-trader/.venv/bin/python


In [2]:
import altair as alt
import pandas as pd

from data_pipeline.src.config import MASTER_CSV_PATH

market = pd.read_csv(MASTER_CSV_PATH, dtype={"symbol": "string"})
market["date"] = pd.to_datetime(market["date"], errors="coerce")

In [3]:
pd.Series({
    "rows": len(market),
    "symbols": market["symbol"].nunique(),
    "earliest_date": market["date"].min(),
    "latest_date": market["date"].max(),
})

rows                           20845
symbols                          818
earliest_date    2026-06-12 00:00:00
latest_date      2026-07-30 00:00:00
dtype: object

In [4]:
market.isna().sum().sort_values(ascending=False).to_frame("missing_rows")

,missing_rows
symbol,0
date,0
ldcp,0
open,0
high,0
low,0
close,0
change,0
change_percent,0
volume,0


In [5]:
history_lengths = (
    market.groupby("symbol", as_index=False)["date"]
    .nunique()
    .rename(columns={"date": "trading_days"})
    .sort_values("trading_days", ascending=False)
)
history_lengths.describe(include="all")

,symbol,trading_days
count,818,818.000000
unique,818,NaN
top,786,NaN
freq,1,NaN
mean,NaN,25.482885
std,NaN,11.001405
min,NaN,1.000000
25%,NaN,12.000000
50%,NaN,33.000000
75%,NaN,33.000000


In [6]:
selected_symbol = "MCB" if "MCB" in set(market["symbol"]) else str(market["symbol"].iloc[0])
selected = market.loc[market["symbol"] == selected_symbol].sort_values("date")
alt.Chart(selected).mark_line().encode(
    x=alt.X("date:T", title="Trading date"),
    y=alt.Y("close:Q", title="Close (PKR)"),
    tooltip=["symbol:N", "date:T", "close:Q"],
).properties(title=f"{selected_symbol} close history", width=800, height=350)

alt.Chart(...)